# 00 · Build the RVL-CDIP working copy

**AIN 320 final project, Phase 1 (PRD §4.3, §5.1, Checklist Step 1).**

This notebook runs **once** for the whole team. It streams the 38.8 GB RVL-CDIP archive from
Hugging Face and keeps only the images in our committed split lists (`splits/`), as small
256×256 grayscale PNGs packed into shards on Google Drive. The raw archive is never stored.

| | |
|---|---|
| Runtime | **CPU is enough** (save your GPU quota for training) |
| Time | roughly 1–2 hours |
| Drive space | ~2–3 GB for Main + test (~4 GB with Large) |
| Disconnected? | Re-run all cells. Finished shards are skipped, so at most one shard (~4,000 images) is redone. |

All logic lives in `src/data/`. This notebook only calls it.

## 1 · Settings

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"   # TODO: your team's GitHub repo
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/ain320_doccls"                # shared project folder on Drive

# The archive is read once, so decide now: Large (5,000/class) adds 48k images (~+1.5 GB on Drive).
INCLUDE_LARGE = False

## 2 · Mount Drive, get the code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone the repo (first run) or pull the latest code, then install dependencies.
import os
REPO_DIR = "/content/final-project"
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 3 · Check the committed split lists

These were generated with `python -m src.data.make_splits` (seed 42) and committed, so every model and every
teammate uses identical images. Dev ⊂ Main ⊂ Large; the test list is the full official test split.

In [ ]:
import json
summary = json.load(open("splits/summary.json"))
print("seed:", summary["seed"])
print("official split sizes:", {k: v["total"] for k, v in summary["official"].items()})
print("our lists:", summary["lists"])

## 4 · Stream the archive and build the working copy

Progress is logged every minute (GB read, % of archive). Expect a long, quiet cell.

In [ ]:
LISTS = "main_train main_val test" + (" large_train" if INCLUDE_LARGE else "")
WC_DIR = f"{DRIVE_ROOT}/working_copy"
!python -m src.data.build_working_copy --out-dir {WC_DIR} --lists {LISTS}

## 5 · Verify

Unpack to local disk exactly as the training notebook will, and check that every image in the lists is present
(PRD checklist: "verify counts per class/split"). A handful of corrupt archive images may be reported missing.
They are listed in `corrupt.csv` and skipped by the data loader.

In [ ]:
WC_LOCAL = "/content/data/working_copy"
!python -m src.data.local_copy unpack --src {WC_DIR} --dst {WC_LOCAL}
!python -m src.data.local_copy verify --data-dir {WC_LOCAL} --lists {LISTS}

In [ ]:
# Record these numbers for the report (PRD Step 1: time taken and working-copy size).
build_log = json.load(open(f"{WC_DIR}/build_log.json"))
for key in ("wanted", "in_manifest", "corrupt", "missing", "working_copy_gb", "total_minutes"):
    print(f"{key:16s} {build_log[key]}")
if os.path.exists(f"{WC_DIR}/corrupt.csv"):
    print(open(f"{WC_DIR}/corrupt.csv").read())

## 6 · Quick look at the data

One sample per class, and the original page sizes logged during extraction (Dataset section of the report).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from src.data.labels import CLASSES

manifest = pd.read_csv(f"{WC_LOCAL}/manifest.csv")
print(manifest.groupby(["split"]).size())

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
train_rows = manifest[manifest.split == "train"]
for label, ax in enumerate(axes.flat):
    row = train_rows[train_rows.label == label].iloc[0]
    ax.imshow(Image.open(f"{WC_LOCAL}/{row.path}"), cmap="gray")
    ax.set_title(CLASSES[label], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
manifest.orig_width.plot.hist(bins=40, ax=axes[0], title="original width (px)")
manifest.orig_height.plot.hist(bins=40, ax=axes[1], title="original height (px)")
plt.tight_layout()
plt.show()